# 장애인콜택시 대기시간 예측 모델링 - LightGBM

이 노트북은 VSCode에서 현재 프로젝트 폴더 기준으로 전처리 CSV를 불러와 `접수→승차 대기시간`을 예측하는 LGBMRegressor 모델링 노트북이다.

## 목적

- 프로젝트 폴더의 `data/processed`에서 전처리 CSV 불러오기
- 임차택시 바로콜 / 특장차 바로콜 승차완료 데이터 생성
- 예측 대상 `target_min = 접수→승차 대기시간` 생성
- 130분 이후 장시간 대기와 02~06시 새벽 호출을 별도관리 구간으로 분리
- 기존과 동일하게 train / validation / test 분리
- LGBMRegressor 기반 모델 학습 및 RF/XGBoost/HGB 결과와 비교

## 모델링 기준

전체 데이터는 보존하되, 일반 패턴 모델에서는 아래 조건을 제외한 버전으로 학습한다.

```text
1. 접수→승차 대기시간 130분 초과
2. 접수시간 02~06시
3. 위 두 조건이 겹치는 데이터
```

주의: LightGBM은 대용량 데이터와 많은 피처에서 빠르게 학습할 수 있는 Gradient Boosting 계열 모델이다.

## 1. 라이브러리 및 로컬 프로젝트 경로 설정

VSCode에서 이 노트북을 실행할 때는 Google Drive 경로를 사용하지 않는다.

현재 프로젝트 구조를 기준으로 아래 CSV를 불러온다.

```text
data/processed/임차택시_대기시간_전처리.csv
data/processed/특장차_대기시간_전처리_접수유형분류.csv
```

노트북 실행 위치가 프로젝트 루트이든 `notebooks_waiting_time` 폴더이든 자동으로 프로젝트 루트를 찾도록 설정한다.

In [1]:
from pathlib import Path
import unicodedata
import platform

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


# 한글 폰트 설정
if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False


In [2]:
# =========================
# 로컬 프로젝트 경로 설정
# =========================

RENTAL_FILENAME = "임차택시_대기시간_전처리.csv"
SPECIAL_FILENAME = "특장차_대기시간_전처리_접수유형분류.csv"


def normalize_text(text):
    return unicodedata.normalize("NFC", str(text))


def find_project_root(start_path=None):
    """
    현재 실행 위치에서 위로 올라가며 data/processed 폴더가 있는 프로젝트 루트를 찾는다.
    """
    start = Path.cwd() if start_path is None else Path(start_path)
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "data/processed 폴더를 찾을 수 없습니다. "
        "VSCode의 현재 작업 폴더가 calltaxi-DA 프로젝트 안인지 확인하세요."
    )


def find_csv_in_processed(processed_dir, filename):
    """
    1순위: data/processed/{filename}
    2순위: 한글 자모 정규화 차이를 고려해 processed 폴더의 CSV 전체에서 파일명 비교
    """
    direct_path = processed_dir / filename

    if direct_path.exists():
        return direct_path

    target_name = normalize_text(filename)
    matches = [
        path
        for path in processed_dir.rglob("*.csv")
        if normalize_text(path.name) == target_name
    ]

    if len(matches) == 0:
        print("processed 폴더에서 찾은 CSV 파일:")
        for path in sorted(processed_dir.rglob("*.csv")):
            print(" -", path)
        raise FileNotFoundError(
            f"{filename} 파일을 찾을 수 없습니다. "
            f"{processed_dir} 안의 파일명을 확인하세요."
        )

    if len(matches) > 1:
        print(f"{filename} 후보가 여러 개입니다. 첫 번째 파일을 사용합니다.")
        for path in matches:
            print(" -", path)

    return matches[0]


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

RENTAL_PATH = find_csv_in_processed(DATA_DIR, RENTAL_FILENAME)
SPECIAL_PATH = find_csv_in_processed(DATA_DIR, SPECIAL_FILENAME)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RENTAL_PATH:", RENTAL_PATH)
print("SPECIAL_PATH:", SPECIAL_PATH)
print("RENTAL exists:", RENTAL_PATH.exists())
print("SPECIAL exists:", SPECIAL_PATH.exists())

PROJECT_ROOT: /Users/blaumonde/calltaxi-DA
DATA_DIR: /Users/blaumonde/calltaxi-DA/data/processed
RENTAL_PATH: /Users/blaumonde/calltaxi-DA/data/processed/임차택시_대기시간_전처리.csv
SPECIAL_PATH: /Users/blaumonde/calltaxi-DA/data/processed/특장차_대기시간_전처리_접수유형분류.csv
RENTAL exists: True
SPECIAL exists: True


## 2. 데이터 로드

모델링 대상은 기존과 동일하게 바로콜 승차완료 데이터만 사용한다.

- 임차택시: `임차택시_바로콜여부 == True`, `대기시간분석_포함여부 == True`
- 특장차: `특장차_바로콜_후보여부 == True` 또는 `특장차_접수유형_후보_최종 == "바로콜 후보"`

In [3]:
SEOUL_GU = {
    "강남구", "강동구", "강북구", "강서구", "관악구",
    "광진구", "구로구", "금천구", "노원구", "도봉구",
    "동대문구", "동작구", "마포구", "서대문구", "서초구",
    "성동구", "성북구", "송파구", "양천구", "영등포구",
    "용산구", "은평구", "종로구", "중구", "중랑구",
}


def existing_cols(path, wanted_cols):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in wanted_cols if col in header]


def classify_move_type(row):
    origin = row.get("출발구")
    dest = row.get("목적구")

    origin_in_seoul = origin in SEOUL_GU
    dest_in_seoul = dest in SEOUL_GU

    if origin_in_seoul and dest_in_seoul:
        if origin == dest:
            return "구 내 이동"
        return "구 간 이동"

    if origin_in_seoul and not dest_in_seoul:
        return "서울→서울 외"

    if not origin_in_seoul and dest_in_seoul:
        return "서울 외→서울"

    return "서울 외↔서울 외"


def load_modeling_dataset():
    common_cols = [
        "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
        "출발구", "출발동", "목적구", "목적동",
        "이용목적", "요금", "승차거리", "차량구분", "장애유형",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "접수_취소_분", "배차_취소_분", "접수승차_날짜차이",
    ]

    rental_cols = common_cols + [
        "예약목적여부",
        "임차택시_바로콜여부",
        "임차택시_장시간예외여부",
        "임차택시_예약성예외여부",
        "임차택시_취소분석유형",
        "대기시간분석_포함여부",
        "대기시간분석_제외사유",
    ]

    special_cols = common_cols + [
        "접수시간대", "접수시간대_HH", "접수요일", "평일주말",
        "세부이동유형", "승차거리_km", "승차거리구간",
        "특장차_접수유형", "특장차_접수유형_분류상태", "특장차_접수유형_메모",
        "접수일자", "예정일자", "취소일자", "접수시", "예정시", "예정시간",
        "취소_접수유형_후보", "_원자료_index",
        "특장차_탑승완료_필수일시존재여부",
        "특장차_탑승완료_시간논리정상여부",
        "심야시간사전예약_후보여부",
        "전일접수_후보여부",
        "특장차_바로콜_후보여부",
        "특장차_접수유형_후보_보완",
        "정기접수_목적후보여부",
        "정기접수_가능패턴여부",
        "동일패턴건수_보완",
        "예정_배차_분", "예정_승차_분",
        "특장차_접수유형_후보_최종",
    ]

    rental = pd.read_csv(
        RENTAL_PATH,
        usecols=existing_cols(RENTAL_PATH, rental_cols),
        low_memory=False,
    )

    special = pd.read_csv(
        SPECIAL_PATH,
        usecols=existing_cols(SPECIAL_PATH, special_cols),
        low_memory=False,
    )

    print("임차택시 원본 shape:", rental.shape)
    print("특장차 원본 shape:", special.shape)

    # datetime 변환
    for frame in [rental, special]:
        for col in [
            "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
            "접수일자", "예정일자", "취소일자",
        ]:
            if col in frame.columns:
                frame[col] = pd.to_datetime(frame[col], errors="coerce")

    # 임차택시 바로콜 승차완료
    rental_model = rental[
        rental["접수일시"].notna()
        & rental["승차일시"].notna()
        & rental["임차택시_바로콜여부"].fillna(False).astype(bool)
        & rental["대기시간분석_포함여부"].fillna(False).astype(bool)
    ].copy()

    rental_model["model_group"] = "임차택시_바로콜"

    # 특장차 바로콜 승차완료
    special_model = special[
        special["접수일시"].notna()
        & special["승차일시"].notna()
        & (
            special["특장차_바로콜_후보여부"].fillna(False).astype(bool)
            | special["특장차_접수유형_후보_최종"].eq("바로콜 후보")
        )
    ].copy()

    special_model["model_group"] = "특장차_바로콜"

    print("임차택시 바로콜 승차완료:", rental_model.shape)
    print("특장차 바로콜 승차완료:", special_model.shape)

    # 컬럼 맞춰서 결합
    all_cols = sorted(set(rental_model.columns) | set(special_model.columns))

    data = pd.concat(
        [
            rental_model.reindex(columns=all_cols),
            special_model.reindex(columns=all_cols),
        ],
        ignore_index=True,
    )

    # numeric 변환
    numeric_cols = [
        "요금", "승차거리", "승차거리_km",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "접수_취소_분", "배차_취소_분", "접수승차_날짜차이",
        "접수시간대_HH", "접수시", "예정시",
        "예정_배차_분", "예정_승차_분", "동일패턴건수_보완",
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    # target 생성
    data = data[
        data["접수일시"].notna()
        & data["승차일시"].notna()
        & data["접수_승차_분"].notna()
        & data["접수_승차_분"].ge(0)
    ].copy()

    data["target_min"] = data["접수_승차_분"]

    # 시간 변수
    data["hour"] = data["접수일시"].dt.hour.astype("int16")
    data["dayofweek"] = data["접수일시"].dt.dayofweek.astype("int16")
    data["month"] = data["접수일시"].dt.month.astype("int16")

    data["is_weekend"] = data["dayofweek"].isin([5, 6]).astype("int8")
    data["is_night"] = data["hour"].between(0, 6, inclusive="both").astype("int8")
    data["is_dawn_02_06"] = data["hour"].between(2, 6, inclusive="both").astype("int8")
    data["is_commute"] = (
        data["hour"].between(7, 9, inclusive="both")
        | data["hour"].between(17, 19, inclusive="both")
    ).astype("int8")

    # 승차거리_km 통일
    if "승차거리_km" not in data.columns:
        data["승차거리_km"] = np.nan

    if "승차거리" in data.columns:
        data["승차거리_km"] = data["승차거리_km"].fillna(
            pd.to_numeric(data["승차거리"], errors="coerce")
        )

    # 세부이동유형 보완
    computed_move_type = data.apply(classify_move_type, axis=1)

    if "세부이동유형" in data.columns:
        data["세부이동유형"] = data["세부이동유형"].fillna(computed_move_type)
        data.loc[data["세부이동유형"].astype(str).eq(""), "세부이동유형"] = computed_move_type
    else:
        data["세부이동유형"] = computed_move_type

    # 문자열 결측 처리
    object_cols = data.select_dtypes(include="object").columns

    for col in object_cols:
        data[col] = data[col].fillna("미상").astype(str)

    data = data.sort_values("접수일시").reset_index(drop=True)

    return data

## 3. 데이터 로드 실행

In [4]:
data = load_modeling_dataset()

print("최종 data shape:", data.shape)
display(data.head())

임차택시 원본 shape: (328875, 28)
특장차 원본 shape: (1393534, 51)
임차택시 바로콜 승차완료: (305729, 29)
특장차 바로콜 승차완료: (1099084, 52)


/var/folders/br/x7f7fw1907z6wpb38fpn1kpm0000gn/T/ipykernel_35759/3787358709.py:189: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = data.select_dtypes(include="object").columns


최종 data shape: (1404813, 67)


,_원자료_index,model_group,대기시간분석_제외사유,대기시간분석_포함여부,동일패턴건수_보완,목적구,목적동,배차_승차_분,배차_취소_분,배차일시,세부이동유형,승차거리,승차거리_km,승차거리구간,승차일시,심야시간사전예약_후보여부,예약목적여부,예정_배차_분,예정_승차_분,예정시,예정시간,예정일시,예정일자,요금,이용목적,임차택시_바로콜여부,임차택시_예약성예외여부,임차택시_장시간예외여부,임차택시_취소분석유형,장애유형,전일접수_후보여부,접수_배차_분,접수_승차_분,접수_취소_분,접수승차_날짜차이,접수시,접수시간대,접수시간대_HH,접수요일,접수일시,접수일자,정기접수_가능패턴여부,정기접수_목적후보여부,차량구분,출발구,출발동,취소_접수유형_후보,취소일시,취소일자,특장차_바로콜_후보여부,특장차_접수유형,특장차_접수유형_메모,특장차_접수유형_분류상태,특장차_접수유형_후보_보완,특장차_접수유형_후보_최종,특장차_탑승완료_시간논리정상여부,특장차_탑승완료_필수일시존재여부,평일주말,하차일시,target_min,hour,dayofweek,month,is_weekend,is_night,is_dawn_02_06,is_commute
0,0.0,특장차_바로콜,미상,미상,0.0,강북구,수유제2동,22.361000,NaN,2025-01-01 00:27:31.307,구 간 이동,17667,17.667,15~25km,2025-01-01 00:49:52.967,False,미상,26.086333,48.447333,0.0,00:01:26,2025-01-01 00:01:26.127,2025-01-01 00:00:00,3400.0,기타,미상,미상,미상,미상,지체,False,26.086333,48.447333,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:01:26.127,2025-01-01 00:00:00,False,False,특장차,용산구,남영동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:35:06.747,48.447333,0,2,1,0,1,0,0
1,2.0,특장차_바로콜,미상,미상,0.0,노원구,하계1동,16.608883,NaN,2025-01-01 00:19:41.900,구 내 이동,2910,2.910,0.5~3km,2025-01-01 00:36:18.433,False,미상,15.698333,32.307217,0.0,00:04:00,2025-01-01 00:04:00.000,2025-01-01 00:00:00,1500.0,기타,미상,미상,미상,미상,뇌병,False,16.177833,32.786717,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:03:31.230,2025-01-01 00:00:00,False,False,특장차,노원구,상계5동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 00:49:45.857,32.786717,0,2,1,0,1,0,0
2,3.0,특장차_바로콜,미상,미상,0.0,중구,명동,20.113167,NaN,2025-01-01 00:33:41.650,구 간 이동,10421,10.421,10~15km,2025-01-01 00:53:48.440,False,미상,29.694167,49.807333,0.0,00:04:00,2025-01-01 00:04:00.000,2025-01-01 00:00:00,2900.0,기타,미상,미상,미상,미상,지체,False,30.017950,50.131117,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:03:40.573,2025-01-01 00:00:00,False,False,특장차,영등포구,당산제1동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:33:38.120,50.131117,0,2,1,0,1,0,0
3,4.0,특장차_바로콜,미상,미상,0.0,서초구,내곡동,33.567500,NaN,2025-01-01 00:06:33.790,구 간 이동,11943,11.943,10~15km,2025-01-01 00:40:07.840,False,미상,2.311883,35.879383,0.0,00:04:15,2025-01-01 00:04:15.077,2025-01-01 00:00:00,3000.0,기타,미상,미상,미상,미상,지체,False,2.311883,35.879383,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:04:15.077,2025-01-01 00:00:00,False,False,특장차,송파구,오금동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:10:15.110,35.879383,0,2,1,0,1,0,0
4,5.0,특장차_바로콜,미상,미상,1.0,서초구,반포2동,38.008050,NaN,2025-01-01 00:16:19.457,구 내 이동,5205,5.205,5~10km,2025-01-01 00:54:19.940,False,미상,10.324283,48.332333,0.0,00:06:00,2025-01-01 00:06:00.000,2025-01-01 00:00:00,1700.0,귀가,미상,미상,미상,미상,뇌병,False,10.424283,48.432333,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:05:54.000,2025-01-01 00:00:00,False,True,특장차,서초구,서초3동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:11:02.643,48.432333,0,2,1,0,1,0,0


In [5]:
display(
    data.groupby("model_group")["target_min"]
    .agg(
        건수="count",
        평균="mean",
        중앙값="median",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        최댓값="max",
    )
    .round(2)
)

,건수,평균,중앙값,p75,p90,p95,p99,최댓값
model_group,,,,,,,,
임차택시_바로콜,305729,42.45,28.51,50.28,87.34,145.05,174.64,249.90
특장차_바로콜,1099084,46.27,33.42,57.33,95.13,133.94,166.64,199.99


## 4. 일반구간 모델링 데이터 생성

전체 데이터는 삭제하지 않고 보존한다.

다만 일반적인 대기시간 예측 모델은 아래 구간을 학습 대상에서 제외한 데이터로 만든다.

```text
제외 조건
1. 접수→승차 대기시간이 130분 초과
2. 접수시간대가 02~06시

In [6]:
# =========================
# 4. 일반구간 / 별도관리 데이터 분리
# =========================

long_wait_cutoff = 130

is_long_tail = data["target_min"].gt(long_wait_cutoff)
is_dawn_02_06 = data["hour"].between(2, 6, inclusive="both")

general_mask = ~is_long_tail & ~is_dawn_02_06

general_model_data = data[general_mask].copy()
excluded_model_data = data[~general_mask].copy()

print("전체 데이터:", data.shape)
print("일반구간 모델링 데이터:", general_model_data.shape)
print("별도관리 데이터:", excluded_model_data.shape)
print("별도관리 비율(%):", round(len(excluded_model_data) / len(data) * 100, 2))

전체 데이터: (1404813, 67)
일반구간 모델링 데이터: (1293478, 67)
별도관리 데이터: (111335, 67)
별도관리 비율(%): 7.93


In [7]:
# =========================
# 4-1. 제외 조건별 건수 확인
# =========================

exclude_summary = pd.Series({
    "전체 건수": len(data),
    "130분 초과 건수": is_long_tail.sum(),
    "02~06시 건수": is_dawn_02_06.sum(),
    "130분 초과 & 02~06시 건수": (is_long_tail & is_dawn_02_06).sum(),
    "130분 초과 비율(%)": is_long_tail.mean() * 100,
    "02~06시 비율(%)": is_dawn_02_06.mean() * 100,
    "겹침 비율 - 전체 대비(%)": (is_long_tail & is_dawn_02_06).mean() * 100,
    "별도관리 전체 비율(%)": (~general_mask).mean() * 100,
})

exclude_summary.round(2)

전체 건수                  1404813.00
130분 초과 건수               78429.00
02~06시 건수                62985.00
130분 초과 & 02~06시 건수      30079.00
130분 초과 비율(%)                5.58
02~06시 비율(%)                 4.48
겹침 비율 - 전체 대비(%)             2.14
별도관리 전체 비율(%)                7.93
dtype: float64

In [8]:
# =========================
# 4-2. 차량유형별 일반구간 / 별도관리 구분 확인
# =========================

data_exclude_check = data.copy()

data_exclude_check["모델링구분"] = np.where(
    general_mask,
    "일반구간_모델링",
    "별도관리_제외",
)

data_exclude_check["제외사유"] = np.select(
    [
        is_long_tail & is_dawn_02_06,
        is_long_tail,
        is_dawn_02_06,
    ],
    [
        "130분초과_AND_02~06시",
        "130분초과",
        "02~06시",
    ],
    default="일반구간",
)

exclude_by_group = (
    data_exclude_check
    .groupby(["model_group", "제외사유"])
    .agg(
        건수=("target_min", "size"),
        평균=("target_min", "mean"),
        중앙값=("target_min", "median"),
        p90=("target_min", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

exclude_by_group["전체대비비율(%)"] = exclude_by_group["건수"] / len(data) * 100

display(exclude_by_group.round(2))

,model_group,제외사유,건수,평균,중앙값,p90,전체대비비율(%)
0,임차택시_바로콜,02~06시,6667,58.32,55.86,96.93,0.47
1,임차택시_바로콜,130분초과,10767,151.95,148.25,171.70,0.77
2,임차택시_바로콜,130분초과_AND_02~06시,10047,162.81,158.17,192.16,0.72
3,임차택시_바로콜,일반구간,278248,33.49,26.61,65.97,19.81
4,특장차_바로콜,02~06시,26239,59.67,53.13,110.85,1.87
5,특장차_바로콜,130분초과,37583,151.28,149.14,169.18,2.68
6,특장차_바로콜,130분초과_AND_02~06시,20032,160.92,158.55,184.48,1.43
7,특장차_바로콜,일반구간,1015230,39.77,31.72,78.40,72.27


In [9]:
# =========================
# 4-3. 일반구간 모델링 데이터 분포 확인
# =========================

display(
    general_model_data
    .groupby("model_group")["target_min"]
    .agg(
        건수="count",
        평균="mean",
        중앙값="median",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        최댓값="max",
    )
    .round(2)
)

,건수,평균,중앙값,p75,p90,p95,최댓값
model_group,,,,,,,
임차택시_바로콜,278248,33.49,26.61,42.41,65.97,79.68,129.98
특장차_바로콜,1015230,39.77,31.72,50.62,78.40,94.12,130.00


## 5. train / validation / test 분리

기존 모델링과 동일하게 데이터를 70% / 15% / 15%로 나눈다.

분리 기준도 기존과 동일하게 `month × model_group`을 stratify 기준으로 사용한다.

이 방식은 월별 분포와 임차택시/특장차 비율이 train / validation / test에 비슷하게 유지되도록 하기 위한 것이다.

이번 모델에서는 일반구간 모델링 데이터인 `general_model_data`를 기준으로 분리한다.

In [10]:
# =========================
# 5. train / validation / test split 함수
# =========================

from sklearn.model_selection import train_test_split

def split_train_valid_test(frame):
    split_key = frame["month"].astype(str) + "_" + frame["model_group"].astype(str)

    train_df, temp_df = train_test_split(
        frame,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=split_key,
    )

    temp_split_key = temp_df["month"].astype(str) + "_" + temp_df["model_group"].astype(str)

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=temp_split_key,
    )

    train_df = train_df.sort_values("접수일시").reset_index(drop=True)
    valid_df = valid_df.sort_values("접수일시").reset_index(drop=True)
    test_df = test_df.sort_values("접수일시").reset_index(drop=True)

    return train_df, valid_df, test_df

In [11]:
# =========================
# 5-1. 일반구간 데이터 split
# =========================

train, valid, test = split_train_valid_test(general_model_data)

print("train:", train.shape)
print("valid:", valid.shape)
print("test:", test.shape)

train: (905434, 67)
valid: (194022, 67)
test: (194022, 67)


In [12]:
# =========================
# 5-2. split 결과 요약
# =========================

def split_summary(frame, name):
    return {
        "split": name,
        "rows": len(frame),
        "target_mean": frame["target_min"].mean(),
        "target_median": frame["target_min"].median(),
        "target_p90": frame["target_min"].quantile(0.90),
        "target_max": frame["target_min"].max(),
        "min_date": frame["접수일시"].min(),
        "max_date": frame["접수일시"].max(),
    }


split_summary_df = pd.DataFrame([
    split_summary(train, "train"),
    split_summary(valid, "valid"),
    split_summary(test, "test"),
])

display(split_summary_df.round(4))

/var/folders/br/x7f7fw1907z6wpb38fpn1kpm0000gn/T/ipykernel_35759/2356030364.py:24: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(split_summary_df.round(4))


,split,rows,target_mean,target_median,target_p90,target_max,min_date,max_date
0,train,905434,38.4277,30.6120,75.6889,129.9986,2025-01-01 00:01:26.127,2025-12-31 23:31:37.000
1,valid,194022,38.4567,30.6165,75.6732,129.9884,2025-01-01 00:03:40.573,2025-12-31 23:52:59.000
2,test,194022,38.3580,30.5818,75.4802,129.9927,2025-01-01 00:03:31.230,2025-12-31 23:40:56.080


In [13]:
# =========================
# 5-3. model_group 비율 확인
# =========================

model_group_ratio = pd.concat(
    [
        train["model_group"].value_counts(normalize=True).rename("train"),
        valid["model_group"].value_counts(normalize=True).rename("valid"),
        test["model_group"].value_counts(normalize=True).rename("test"),
    ],
    axis=1,
)

display(model_group_ratio.round(4))

,train,valid,test
model_group,,,
특장차_바로콜,0.7849,0.7849,0.7849
임차택시_바로콜,0.2151,0.2151,0.2151


In [14]:
# =========================
# 5-4. month 비율 확인
# =========================

month_ratio = pd.concat(
    [
        train["month"].value_counts(normalize=True).sort_index().rename("train"),
        valid["month"].value_counts(normalize=True).sort_index().rename("valid"),
        test["month"].value_counts(normalize=True).sort_index().rename("test"),
    ],
    axis=1,
)

display(month_ratio.round(4))

,train,valid,test
month,,,
1,0.0745,0.0745,0.0745
2,0.0774,0.0774,0.0774
3,0.0819,0.0819,0.0819
4,0.0852,0.0852,0.0852
5,0.0806,0.0806,0.0806
6,0.0822,0.0822,0.0822
7,0.0945,0.0945,0.0945
8,0.0875,0.0875,0.0875
9,0.0881,0.0881,0.0881


## 6. Feature Set v2: 승차거리 기준 LightGBM 모델링

이번 실험에서는 RF/XGBoost/HGB 모델과 동일한 피처셋을 사용해 LGBMRegressor 모델을 학습한다.

사용 피처는 아래 11개다.

```python
FEATURES_V2 = [
    "hour",
    "이용목적",
    "승차거리",
    "출발동",
    "목적동",
    "출발구",
    "목적구",
    "month",
    "세부이동유형",
    "dayofweek",
    "model_group",
]
```

동일한 train / validation / test split을 사용하므로 RF, XGBoost, HGB 결과와 직접 비교할 수 있다.

In [15]:
# =========================
# 6-1. Feature Set v2 정의
# =========================

FEATURES_V2 = [
    "hour",
    "이용목적",
    "승차거리",
    "출발동",
    "목적동",
    "출발구",
    "목적구",
    "month",
    "세부이동유형",
    "dayofweek",
    "model_group",
]

TARGET_COL = "target_min"

missing_features = [col for col in FEATURES_V2 if col not in train.columns]
if missing_features:
    raise ValueError(f"train에 없는 피처가 있습니다: {missing_features}")

# 승차거리는 모델 입력용 수치형으로 통일
for frame in [train, valid, test]:
    frame["승차거리"] = pd.to_numeric(frame["승차거리"], errors="coerce")

print("FEATURES_V2 개수:", len(FEATURES_V2))
print(FEATURES_V2)

display(train[FEATURES_V2 + [TARGET_COL]].head())

FEATURES_V2 개수: 11
['hour', '이용목적', '승차거리', '출발동', '목적동', '출발구', '목적구', 'month', '세부이동유형', 'dayofweek', 'model_group']


,hour,이용목적,승차거리,출발동,목적동,출발구,목적구,month,세부이동유형,dayofweek,model_group,target_min
0,0,기타,17667,남영동,수유제2동,용산구,강북구,1,구 간 이동,2,특장차_바로콜,48.447333
1,0,기타,4858,상일동,성내제2동,강동구,강동구,1,구 내 이동,2,특장차_바로콜,52.051550
2,0,귀가,5145,중계2.3동,상계1동,노원구,노원구,1,구 내 이동,2,특장차_바로콜,38.206117
3,0,기타,9000,이화동,월계2동,종로구,노원구,1,구 간 이동,2,특장차_바로콜,92.664717
4,0,귀가,9702,여의동,증산동,영등포구,은평구,1,구 간 이동,2,특장차_바로콜,82.073717


## 7. 평가 함수 정의

RF 노트북과 동일한 평가 지표를 사용한다.

- `MAE`: 평균 절대 오차
- `Median_AE`: 중앙 절대 오차
- `RMSE`: 큰 오차에 민감한 지표
- `R2`: 결정계수
- `risk_*`: train set의 90분위수 기준 장시간 대기 여부 평가

In [16]:
# =========================
# 7-1. 평가 함수
# =========================

from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Median_AE": median_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def risk_metrics(y_true, y_pred, threshold):
    actual = y_true >= threshold
    pred = y_pred >= threshold
    return {
        "threshold": threshold,
        "Accuracy": accuracy_score(actual, pred),
        "Precision": precision_score(actual, pred, zero_division=0),
        "Recall": recall_score(actual, pred, zero_division=0),
        "F1": f1_score(actual, pred, zero_division=0),
        "Confusion_Matrix": confusion_matrix(actual, pred, labels=[False, True]).tolist(),
    }


def evaluate_predictions(frame, pred, threshold):
    y = frame["target_min"].to_numpy()
    return {
        **regression_metrics(y, pred),
        **{f"risk_{k}": v for k, v in risk_metrics(y, pred, threshold).items()},
    }

## 8. LGBMRegressor 모델 학습

범주형 변수는 `OrdinalEncoder`로 변환하고, 수치형 변수는 그대로 사용한다.

LightGBM은 대용량 데이터에서 학습 속도가 빠른 Gradient Boosting 계열 모델이다. 현재처럼 90만 건 이상의 데이터와 구·동 단위 범주형 변수가 포함된 피처셋에서 RF/XGBoost/HGB와 비교하기 적합하다.

In [17]:
# =========================
# 8-1. LGBMRegressor Pipeline 구성
# =========================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline

try:
    from lightgbm import LGBMRegressor
except ImportError as e:
    raise ImportError("lightgbm이 설치되어 있지 않습니다. `pip install lightgbm` 후 다시 실행하세요.") from e

CATEGORICAL_COLS_V2 = [
    "이용목적",
    "출발동",
    "목적동",
    "출발구",
    "목적구",
    "세부이동유형",
    "model_group",
]

NUMERIC_COLS_V2 = [
    "hour",
    "승차거리",
    "month",
    "dayofweek",
]


def build_lgbm_v2_pipeline(params=None):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                ),
                CATEGORICAL_COLS_V2,
            ),
            (
                "num",
                "passthrough",
                NUMERIC_COLS_V2,
            ),
        ],
        verbose_feature_names_out=False,
        remainder="drop",
    )

    default_params = {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "objective": "regression",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    }

    if params:
        default_params.update(params)

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", LGBMRegressor(**default_params)),
    ])


In [18]:
# =========================
# 8-2. LGBMRegressor 모델 학습 및 평가
# =========================

long_wait_threshold_v2 = train[TARGET_COL].quantile(0.90)
threshold_by_group_v2 = train.groupby("model_group")[TARGET_COL].quantile(0.90).to_dict()

print("장시간 대기 기준 p90:", round(long_wait_threshold_v2, 4))
print("차량유형별 p90:", {k: round(v, 4) for k, v in threshold_by_group_v2.items()})

lgbm_v2_model = build_lgbm_v2_pipeline()

lgbm_v2_model.fit(
    train[FEATURES_V2],
    train[TARGET_COL],
)

valid_pred_lgbm_v2 = lgbm_v2_model.predict(valid[FEATURES_V2])
test_pred_lgbm_v2 = lgbm_v2_model.predict(test[FEATURES_V2])

valid_metrics_lgbm_v2 = evaluate_predictions(
    valid,
    valid_pred_lgbm_v2,
    threshold=long_wait_threshold_v2,
)

test_metrics_lgbm_v2 = evaluate_predictions(
    test,
    test_pred_lgbm_v2,
    threshold=long_wait_threshold_v2,
)

lgbm_v2_results = pd.DataFrame([
    {"model": "LightGBM Feature Set v2", "split": "valid", **valid_metrics_lgbm_v2},
    {"model": "LightGBM Feature Set v2", "split": "test", **test_metrics_lgbm_v2},
])

display(
    lgbm_v2_results
    .drop(columns=["risk_Confusion_Matrix"])
    .round(4)
)

print("test confusion matrix:")
test_metrics_lgbm_v2["risk_Confusion_Matrix"]


장시간 대기 기준 p90: 75.6889
차량유형별 p90: {'임차택시_바로콜': 66.0455, '특장차_바로콜': 78.443}


,model,split,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,LightGBM Feature Set v2,valid,11.9254,8.7532,16.3528,0.5485,75.6889,0.919,0.7215,0.3079,0.4316
1,LightGBM Feature Set v2,test,11.9504,8.7772,16.3911,0.5441,75.6889,0.919,0.7155,0.3032,0.4259


test confusion matrix:


[[172473, 2318], [13400, 5831]]

## 9. 차량유형별 성능 확인

통합 LightGBM 모델이 임차택시 바로콜과 특장차 바로콜에서 각각 어떻게 동작하는지 따로 확인한다.

In [19]:
# =========================
# 9-1. 차량유형별 성능 평가
# =========================

def evaluate_by_group(frame, pred, threshold_by_group):
    tmp = frame[["model_group", TARGET_COL]].copy()
    tmp["prediction"] = pred

    rows = []
    for group, group_df in tmp.groupby("model_group"):
        y = group_df[TARGET_COL].to_numpy()
        p = group_df["prediction"].to_numpy()
        threshold = threshold_by_group[group]

        reg = regression_metrics(y, p)
        risk = risk_metrics(y, p, threshold)

        rows.append({
            "model_group": group,
            "rows": len(group_df),
            **reg,
            "risk_threshold": threshold,
            "risk_Accuracy": risk["Accuracy"],
            "risk_Precision": risk["Precision"],
            "risk_Recall": risk["Recall"],
            "risk_F1": risk["F1"],
        })

    return pd.DataFrame(rows)


group_metrics_lgbm_v2 = evaluate_by_group(
    test,
    test_pred_lgbm_v2,
    threshold_by_group_v2,
)

display(group_metrics_lgbm_v2.round(4))

,model_group,rows,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,임차택시_바로콜,41738,10.6538,7.8422,14.6062,0.5376,66.0455,0.9196,0.6773,0.3251,0.4393
1,특장차_바로콜,152284,12.3057,9.0670,16.8473,0.5387,78.4430,0.9180,0.7179,0.2821,0.4050


## 10. RF/XGBoost/HGB 결과와 비교

RF, XGBoost, HGB, LightGBM 결과를 비교해 같은 피처셋에서 모델만 바꿨을 때 성능이 개선되는지 확인한다.

비교 시 우선순위는 다음과 같다.

1. `MAE`
2. `Median_AE`
3. `RMSE`
4. `R2`
5. 장시간 대기 지표는 참고용


동일한 Feature Set v2와 동일한 train/validation/test 분리 기준에서 RandomForest, XGBoost, HistGradientBoosting, LightGBM 모델을 비교하였다.

예상 대기시간을 제공하는 목적에서는 MAE를 최우선 지표로 보았다.  
현재 결과 기준으로 RandomForest의 test MAE가 약 11.72분으로 가장 낮게 나타났고, LightGBM과 HGB는 약 11.95분 수준으로 비슷한 성능을 보였다. XGBoost도 큰 차이는 없지만 세 모델 중에서는 약간 낮은 성능을 보였다.

차량유형별로는 임차택시 바로콜의 MAE가 약 10.65분 수준, 특장차 바로콜의 MAE가 약 12.3분 수준으로 나타났다. 즉, 특장차 바로콜이 임차택시보다 예측 난이도가 더 높았다.

장시간 대기 탐지 지표는 전반적으로 Precision에 비해 Recall이 낮았다. 이는 모델이 장시간 대기라고 예측한 경우에는 어느 정도 맞지만, 실제 장시간 대기 사례를 충분히 잡아내지는 못한다는 의미다.

따라서 최종적으로는 예상 대기시간 제공 목적에는 RandomForest를 기준 모델로 사용하고, LightGBM/HGB는 성능 비교 후보로 제시하는 것이 적절하다.

## 11. Proxy 추가: 출발구×요일×시간대 평소 대기시간

MAE 개선 여부를 확인하기 위해 `출발구 × 요일 × 시간대` 조합별 평소 대기시간 proxy를 추가한다.

추가 피처는 다음과 같다.

```python
origin_gu_weekday_hour_wait_p50
```

이 변수는 `target_min`의 중앙값을 이용해 만들기 때문에 target leakage 위험이 있다.  
따라서 전체 데이터를 한 번에 집계해서 붙이지 않고, 아래 방식으로 생성한다.

- train set: OOF 방식으로 생성
  - 각 fold의 값은 자기 fold를 제외한 나머지 train fold에서만 계산
  - 각 행이 자기 자신의 `target_min`을 직접 참조하지 않도록 함
- validation/test set: train 전체에서 계산한 통계값만 merge
  - validation/test의 정답 정보는 proxy 생성에 사용하지 않음

주의: 실제 서비스 적용 시에는 예측 시점 이전의 누적 이력 데이터만 사용해 동일한 통계를 계산해야 한다.

In [20]:
# =========================
# 11-1. OOF target-stat proxy 생성 함수
# =========================

from sklearn.model_selection import StratifiedKFold, KFold


def add_oof_group_target_stat(
    train_df,
    valid_df,
    test_df,
    group_cols,
    target_col,
    stat_name,
    agg_func="median",
    n_splits=5,
    random_state=42,
):
    """target 기반 group 통계 proxy를 leakage 방지 방식으로 생성한다.

    train:
        OOF 방식으로 각 fold의 proxy를 나머지 fold에서 계산한다.
    valid/test:
        train 전체에서 계산한 통계표만 merge한다.
    """
    train_result = train_df.copy()
    valid_result = valid_df.copy()
    test_result = test_df.copy()

    global_stat = train_result[target_col].agg(agg_func)
    train_result[stat_name] = np.nan

    if {"month", "model_group"}.issubset(train_result.columns):
        strata = train_result["month"].astype(str) + "_" + train_result["model_group"].astype(str)
        splitter = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )
        split_iter = splitter.split(train_result, strata)
    else:
        splitter = KFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )
        split_iter = splitter.split(train_result)

    for fold_idx, (fit_idx, oof_idx) in enumerate(split_iter, start=1):
        fold_fit = train_result.iloc[fit_idx]
        fold_oof_keys = train_result.iloc[oof_idx][group_cols]

        stat_table = (
            fold_fit
            .groupby(group_cols, observed=True)[target_col]
            .agg(agg_func)
            .reset_index()
            .rename(columns={target_col: stat_name})
        )

        mapped = fold_oof_keys.merge(
            stat_table,
            on=group_cols,
            how="left",
        )[stat_name].to_numpy()

        train_result.iloc[oof_idx, train_result.columns.get_loc(stat_name)] = mapped
        print(f"fold {fold_idx}/{n_splits} 완료")

    train_result[stat_name] = train_result[stat_name].fillna(global_stat)

    full_stat_table = (
        train_df
        .groupby(group_cols, observed=True)[target_col]
        .agg(agg_func)
        .reset_index()
        .rename(columns={target_col: stat_name})
    )

    valid_result = valid_result.merge(full_stat_table, on=group_cols, how="left")
    test_result = test_result.merge(full_stat_table, on=group_cols, how="left")

    valid_result[stat_name] = valid_result[stat_name].fillna(global_stat)
    test_result[stat_name] = test_result[stat_name].fillna(global_stat)

    return train_result, valid_result, test_result, full_stat_table


In [21]:
# =========================
# 11-2. 출발구 × 요일 × 시간대 p50 proxy 생성
# =========================

ORIGIN_GU_WEEKDAY_HOUR_COLS = [
    "출발구",
    "dayofweek",
    "hour",
]

ORIGIN_GU_WEEKDAY_HOUR_P50_COL = "origin_gu_weekday_hour_wait_p50"

train_p50, valid_p50, test_p50, origin_gu_weekday_hour_wait_p50_table = (
    add_oof_group_target_stat(
        train,
        valid,
        test,
        group_cols=ORIGIN_GU_WEEKDAY_HOUR_COLS,
        target_col=TARGET_COL,
        stat_name=ORIGIN_GU_WEEKDAY_HOUR_P50_COL,
        agg_func="median",
        n_splits=5,
        random_state=RANDOM_STATE,
    )
)

print("proxy 생성 후 shape:")
print("train_p50:", train_p50.shape)
print("valid_p50:", valid_p50.shape)
print("test_p50:", test_p50.shape)

print("proxy 결측치 수:")
print("train:", train_p50[ORIGIN_GU_WEEKDAY_HOUR_P50_COL].isna().sum())
print("valid:", valid_p50[ORIGIN_GU_WEEKDAY_HOUR_P50_COL].isna().sum())
print("test:", test_p50[ORIGIN_GU_WEEKDAY_HOUR_P50_COL].isna().sum())

display(
    train_p50[
        ORIGIN_GU_WEEKDAY_HOUR_COLS
        + [ORIGIN_GU_WEEKDAY_HOUR_P50_COL, TARGET_COL]
    ].head()
)

display(
    origin_gu_weekday_hour_wait_p50_table
    .sort_values(ORIGIN_GU_WEEKDAY_HOUR_P50_COL, ascending=False)
    .head(20)
    .round(4)
)


fold 1/5 완료
fold 2/5 완료
fold 3/5 완료
fold 4/5 완료
fold 5/5 완료
proxy 생성 후 shape:
train_p50: (905434, 68)
valid_p50: (194022, 68)
test_p50: (194022, 68)
proxy 결측치 수:
train: 0
valid: 0
test: 0


,출발구,dayofweek,hour,origin_gu_weekday_hour_wait_p50,target_min
0,용산구,2,0,29.332450,48.447333
1,강동구,2,0,49.637358,52.051550
2,노원구,2,0,23.979117,38.206117
3,종로구,2,0,39.193067,92.664717
4,영등포구,2,0,48.674667,82.073717


,출발구,dayofweek,hour,origin_gu_weekday_hour_wait_p50
2553,성남시분당구,0,17,129.7478
2221,부천시오정구,4,18,129.6338
2270,부천시원미구,5,14,129.5682
1492,남양주시,6,9,129.5570
704,과천시,2,13,129.4106
2545,서초구,6,22,128.9359
2641,성남시수정구,5,16,128.7950
602,고양시덕양구,6,17,128.7722
3118,안양시만안구,6,12,128.6480
3105,안양시만안구,3,9,128.5662


## 12. LGBMRegressor + 출발구×요일×시간대 p50 proxy 재학습

기존 Feature Set v2에 `origin_gu_weekday_hour_wait_p50`을 추가해 LGBMRegressor 모델을 다시 학습한다.

비교 기준은 다음과 같다.

1. `MAE`
2. `Median_AE`
3. `RMSE`
4. `R2`
5. `risk_F1`

이번 실험의 핵심 목적은 장시간 대기 탐지보다 `MAE`가 줄어드는지 확인하는 것이다.

In [22]:
# =========================
# 12-1. proxy 추가 Feature Set 정의
# =========================

FEATURES_V2_P50 = FEATURES_V2 + [
    ORIGIN_GU_WEEKDAY_HOUR_P50_COL,
]

NUMERIC_COLS_V2_P50 = NUMERIC_COLS_V2 + [
    ORIGIN_GU_WEEKDAY_HOUR_P50_COL,
]

missing_features = [col for col in FEATURES_V2_P50 if col not in train_p50.columns]
if missing_features:
    raise ValueError(f"train_p50에 없는 피처가 있습니다: {missing_features}")

print("기존 피처 수:", len(FEATURES_V2))
print("p50 proxy 추가 후 피처 수:", len(FEATURES_V2_P50))
print(FEATURES_V2_P50)

display(train_p50[FEATURES_V2_P50 + [TARGET_COL]].head())


기존 피처 수: 11
p50 proxy 추가 후 피처 수: 12
['hour', '이용목적', '승차거리', '출발동', '목적동', '출발구', '목적구', 'month', '세부이동유형', 'dayofweek', 'model_group', 'origin_gu_weekday_hour_wait_p50']


,hour,이용목적,승차거리,출발동,목적동,출발구,목적구,month,세부이동유형,dayofweek,model_group,origin_gu_weekday_hour_wait_p50,target_min
0,0,기타,17667,남영동,수유제2동,용산구,강북구,1,구 간 이동,2,특장차_바로콜,29.332450,48.447333
1,0,기타,4858,상일동,성내제2동,강동구,강동구,1,구 내 이동,2,특장차_바로콜,49.637358,52.051550
2,0,귀가,5145,중계2.3동,상계1동,노원구,노원구,1,구 내 이동,2,특장차_바로콜,23.979117,38.206117
3,0,기타,9000,이화동,월계2동,종로구,노원구,1,구 간 이동,2,특장차_바로콜,39.193067,92.664717
4,0,귀가,9702,여의동,증산동,영등포구,은평구,1,구 간 이동,2,특장차_바로콜,48.674667,82.073717


In [23]:
# =========================
# 12-2. proxy 추가 LightGBM Pipeline 구성
# =========================


def build_lgbm_v2_p50_pipeline(params=None):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                ),
                CATEGORICAL_COLS_V2,
            ),
            (
                "num",
                "passthrough",
                NUMERIC_COLS_V2_P50,
            ),
        ],
        verbose_feature_names_out=False,
        remainder="drop",
    )

    default_params = {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "objective": "regression",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    }

    if params:
        default_params.update(params)

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", LGBMRegressor(**default_params)),
    ])


In [24]:
# =========================
# 12-3. LGBMRegressor + p50 proxy 학습 및 평가
# =========================

lgbm_v2_p50_model = build_lgbm_v2_p50_pipeline()

lgbm_v2_p50_model.fit(
    train_p50[FEATURES_V2_P50],
    train_p50[TARGET_COL],
)

valid_pred_lgbm_v2_p50 = lgbm_v2_p50_model.predict(valid_p50[FEATURES_V2_P50])
test_pred_lgbm_v2_p50 = lgbm_v2_p50_model.predict(test_p50[FEATURES_V2_P50])

valid_metrics_lgbm_v2_p50 = evaluate_predictions(
    valid_p50,
    valid_pred_lgbm_v2_p50,
    threshold=long_wait_threshold_v2,
)

test_metrics_lgbm_v2_p50 = evaluate_predictions(
    test_p50,
    test_pred_lgbm_v2_p50,
    threshold=long_wait_threshold_v2,
)

lgbm_v2_p50_results = pd.DataFrame([
    {"model": "LightGBM Feature Set v2 + origin_gu_weekday_hour_wait_p50", "split": "valid", **valid_metrics_lgbm_v2_p50},
    {"model": "LightGBM Feature Set v2 + origin_gu_weekday_hour_wait_p50", "split": "test", **test_metrics_lgbm_v2_p50},
])

display(
    lgbm_v2_p50_results
    .drop(columns=["risk_Confusion_Matrix"], errors="ignore")
    .round(4)
)

print("test confusion matrix:")
test_metrics_lgbm_v2_p50["risk_Confusion_Matrix"]


,model,split,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,LightGBM Feature Set v2 + origin_gu_weekday_ho...,valid,11.8254,8.6074,16.2798,0.5525,75.6889,0.9199,0.7087,0.3368,0.4566
1,LightGBM Feature Set v2 + origin_gu_weekday_ho...,test,11.8465,8.6042,16.3239,0.5478,75.6889,0.9198,0.6992,0.3353,0.4533


test confusion matrix:


[[172017, 2774], [12782, 6449]]

## 13. 기존 LGBMRegressor 기준 성능과 proxy 추가 성능 비교

기존 Feature Set v2 LGBMRegressor 결과와 `origin_gu_weekday_hour_wait_p50` proxy 추가 결과를 비교한다.

해석 기준은 다음과 같다.

- `MAE`가 줄면 예상 대기시간 제공 성능이 개선된 것
- `Median_AE`가 줄면 일반적인 사용자 체감 오차가 개선된 것
- `RMSE`가 줄면 큰 오차가 줄어든 것
- `risk_F1`은 장시간 대기 탐지 관점의 참고 지표

In [25]:
# =========================
# 13-1. baseline vs p50 proxy 비교
# =========================

proxy_compare = pd.concat(
    [
        lgbm_v2_results.assign(feature_set="Feature Set v2"),
        lgbm_v2_p50_results.assign(feature_set="Feature Set v2 + origin_gu_weekday_hour_wait_p50"),
    ],
    ignore_index=True,
)

compare_cols = [
    "feature_set",
    "model",
    "split",
    "MAE",
    "Median_AE",
    "RMSE",
    "R2",
    "risk_threshold",
    "risk_Accuracy",
    "risk_Precision",
    "risk_Recall",
    "risk_F1",
]

display(
    proxy_compare[compare_cols]
    .sort_values(["split", "MAE"])
    .round(4)
)

baseline_test = lgbm_v2_results.query("split == 'test'").iloc[0]
p50_test = lgbm_v2_p50_results.query("split == 'test'").iloc[0]

improvement_summary = pd.DataFrame([
    {
        "metric": "MAE",
        "baseline": baseline_test["MAE"],
        "p50_proxy": p50_test["MAE"],
        "diff_p50_minus_baseline": p50_test["MAE"] - baseline_test["MAE"],
    },
    {
        "metric": "Median_AE",
        "baseline": baseline_test["Median_AE"],
        "p50_proxy": p50_test["Median_AE"],
        "diff_p50_minus_baseline": p50_test["Median_AE"] - baseline_test["Median_AE"],
    },
    {
        "metric": "RMSE",
        "baseline": baseline_test["RMSE"],
        "p50_proxy": p50_test["RMSE"],
        "diff_p50_minus_baseline": p50_test["RMSE"] - baseline_test["RMSE"],
    },
    {
        "metric": "risk_F1",
        "baseline": baseline_test["risk_F1"],
        "p50_proxy": p50_test["risk_F1"],
        "diff_p50_minus_baseline": p50_test["risk_F1"] - baseline_test["risk_F1"],
    },
])

display(improvement_summary.round(4))


,feature_set,model,split,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
3,Feature Set v2 + origin_gu_weekday_hour_wait_p50,LightGBM Feature Set v2 + origin_gu_weekday_ho...,test,11.8465,8.6042,16.3239,0.5478,75.6889,0.9198,0.6992,0.3353,0.4533
1,Feature Set v2,LightGBM Feature Set v2,test,11.9504,8.7772,16.3911,0.5441,75.6889,0.9190,0.7155,0.3032,0.4259
2,Feature Set v2 + origin_gu_weekday_hour_wait_p50,LightGBM Feature Set v2 + origin_gu_weekday_ho...,valid,11.8254,8.6074,16.2798,0.5525,75.6889,0.9199,0.7087,0.3368,0.4566
0,Feature Set v2,LightGBM Feature Set v2,valid,11.9254,8.7532,16.3528,0.5485,75.6889,0.9190,0.7215,0.3079,0.4316


,metric,baseline,p50_proxy,diff_p50_minus_baseline
0,MAE,11.9504,11.8465,-0.1039
1,Median_AE,8.7772,8.6042,-0.1731
2,RMSE,16.3911,16.3239,-0.0672
3,risk_F1,0.4259,0.4533,0.0274


In [26]:
# =========================
# 13-2. proxy 추가 모델 차량유형별 성능 평가
# =========================

group_metrics_lgbm_v2_p50 = evaluate_by_group(
    test_p50,
    test_pred_lgbm_v2_p50,
    threshold_by_group_v2,
)

display(group_metrics_lgbm_v2_p50.round(4))


,model_group,rows,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,임차택시_바로콜,41738,10.5693,7.7348,14.5598,0.5406,66.0455,0.9205,0.6863,0.3290,0.4448
1,특장차_바로콜,152284,12.1965,8.8775,16.7751,0.5426,78.4430,0.9186,0.6957,0.3142,0.4329


## 14. 해석 메모

`origin_gu_weekday_hour_wait_p50`은 출발구·요일·시간대별 평소 대기시간을 반영하는 proxy이다.

이 피처는 `target_min` 기반이므로 leakage 위험이 있지만, 본 실험에서는 다음 방식으로 방지하였다.

- train set에서는 OOF 방식으로 proxy를 생성했다.
- validation/test set에서는 train set에서 계산한 통계값만 사용했다.
- 따라서 validation/test의 정답 대기시간은 proxy 생성에 사용되지 않았다.

결과 해석 시에는 기존 LGBMRegressor Feature Set v2 대비 MAE가 감소했는지를 가장 우선적으로 확인한다.  
MAE가 감소했다면 지역·요일·시간대별 평소 대기시간 패턴이 예상 대기시간 예측에 도움이 된 것으로 볼 수 있다.

## 15. LGBM + 전일 차량운행 + 날씨 피처 추가

RandomForest에서 성능 개선이 확인된 `전일 차량운행 대수`와 `날씨 피처`를 LGBMRegressor에도 추가해 비교한다.

비교 기준은 다음과 같다.

```text
LightGBM Feature Set v2
LightGBM Feature Set v2 + origin_gu_weekday_hour_wait_p50
LightGBM Feature Set v2 + vehicle_operation_count_prev_day + weather
```

이번 실험에서 사용하는 추가 피처는 다음과 같다.

```python
vehicle_operation_count_prev_day

temperature_c
precipitation_mm
wind_speed_ms
snow_depth_cm
is_bad_weather
```

`vehicle_operation_count_prev_day`는 예측 시점 이전에 확인 가능한 전일 공급 proxy이고, 날씨 피처는 접수시각의 날짜·시간 기준으로 병합한다.

In [27]:
# =========================
# 15-1. 전일 차량운행 table 생성
# =========================

DAILY_USAGE_FILENAME = "서울시설공단_장애인콜택시 일별이용현황_20251231.csv"
DAILY_USAGE_PATH = find_csv_in_processed(DATA_DIR, DAILY_USAGE_FILENAME)

print("DAILY_USAGE_PATH:", DAILY_USAGE_PATH)

daily_usage = pd.read_csv(DAILY_USAGE_PATH, low_memory=False)

required_daily_cols = ["기준일", "차량운행"]
missing_daily_cols = [col for col in required_daily_cols if col not in daily_usage.columns]
if missing_daily_cols:
    raise ValueError(f"일별이용현황 데이터에 필요한 컬럼이 없습니다: {missing_daily_cols}")

daily_usage["기준일"] = pd.to_datetime(daily_usage["기준일"], errors="coerce").dt.normalize()
daily_usage["차량운행"] = pd.to_numeric(daily_usage["차량운행"], errors="coerce")

vehicle_operation_today_table = (
    daily_usage
    .dropna(subset=["기준일"])
    .groupby("기준일", as_index=False)
    .agg(vehicle_operation_count_today=("차량운행", "first"))
)

PREV_DAY_VEHICLE_OPERATION_COL = "vehicle_operation_count_prev_day"

vehicle_operation_prev_day_table = vehicle_operation_today_table.copy()
vehicle_operation_prev_day_table["기준일"] = vehicle_operation_prev_day_table["기준일"] + pd.Timedelta(days=1)
vehicle_operation_prev_day_table = vehicle_operation_prev_day_table.rename(
    columns={"vehicle_operation_count_today": PREV_DAY_VEHICLE_OPERATION_COL}
)

print("전일 차량운행 table shape:", vehicle_operation_prev_day_table.shape)
print("기준일 범위:", vehicle_operation_prev_day_table["기준일"].min(), "~", vehicle_operation_prev_day_table["기준일"].max())

display(vehicle_operation_prev_day_table.head())
display(vehicle_operation_prev_day_table[PREV_DAY_VEHICLE_OPERATION_COL].describe().to_frame().round(2))

DAILY_USAGE_PATH: /Users/blaumonde/calltaxi-DA/data/processed/서울시설공단_장애인콜택시 일별이용현황_20251231.csv
전일 차량운행 table shape: (365, 2)
기준일 범위: 2025-01-02 00:00:00 ~ 2026-01-01 00:00:00


,기준일,vehicle_operation_count_prev_day
0,2025-01-02,257
1,2025-01-03,665
2,2025-01-04,655
3,2025-01-05,407
4,2025-01-06,242


,vehicle_operation_count_prev_day
count,365.00
mean,594.46
std,213.29
min,207.00
25%,340.00
50%,680.00
75%,772.00
max,853.00


In [28]:
# =========================
# 15-2. 날씨 데이터 로드 및 확인
# =========================

WEATHER_FILENAME = "weather_asos_seoul_2025_hourly_clean.csv"
WEATHER_PATH = DATA_DIR / WEATHER_FILENAME

if not WEATHER_PATH.exists():
    raise FileNotFoundError(
        f"날씨 정제 파일이 없습니다: {WEATHER_PATH}. "
        "먼저 notebooks/09_weather_data_cleaning.ipynb를 실행해 정제 CSV를 생성하세요."
    )

weather_hourly = pd.read_csv(
    WEATHER_PATH,
    encoding="utf-8-sig",
    low_memory=False,
)

weather_hourly["weather_date"] = pd.to_datetime(weather_hourly["weather_date"], errors="coerce").dt.date
weather_hourly["weather_hour"] = pd.to_numeric(weather_hourly["weather_hour"], errors="coerce").astype("Int64")

WEATHER_FEATURES_SIMPLE = [
    "temperature_c",
    "precipitation_mm",
    "wind_speed_ms",
    "snow_depth_cm",
    "is_bad_weather",
]

for col in WEATHER_FEATURES_SIMPLE:
    weather_hourly[col] = pd.to_numeric(weather_hourly[col], errors="coerce")

weather_merge_table = (
    weather_hourly[["weather_date", "weather_hour"] + WEATHER_FEATURES_SIMPLE]
    .dropna(subset=["weather_date", "weather_hour"])
    .drop_duplicates(subset=["weather_date", "weather_hour"], keep="first")
    .copy()
)
weather_merge_table["weather_hour"] = weather_merge_table["weather_hour"].astype(int)

print("weather shape:", weather_hourly.shape)
print("weather merge table shape:", weather_merge_table.shape)
print("날짜 범위:", weather_merge_table["weather_date"].min(), "~", weather_merge_table["weather_date"].max())
print("사용 날씨 피처:", WEATHER_FEATURES_SIMPLE)

display(weather_merge_table.head())
display(weather_merge_table[WEATHER_FEATURES_SIMPLE].describe().T.round(4))

weather shape: (8760, 18)
weather merge table shape: (8760, 7)
날짜 범위: 2025-01-01 ~ 2025-12-31
사용 날씨 피처: ['temperature_c', 'precipitation_mm', 'wind_speed_ms', 'snow_depth_cm', 'is_bad_weather']


,weather_date,weather_hour,temperature_c,precipitation_mm,wind_speed_ms,snow_depth_cm,is_bad_weather
0,2025-01-01,0,-1.2,0.0,0.7,0.0,0
1,2025-01-01,1,-1.7,0.0,1.1,0.0,0
2,2025-01-01,2,-1.8,0.0,0.5,0.0,0
3,2025-01-01,3,-2.0,0.0,1.9,0.0,0
4,2025-01-01,4,-2.3,0.0,2.2,0.0,0


,count,mean,std,min,25%,50%,75%,max
temperature_c,8760.0,14.1435,11.2933,-12.1,4.5,15.0,23.9,37.6
precipitation_mm,8760.0,0.1806,1.3175,0.0,0.0,0.0,0.0,35.2
wind_speed_ms,8760.0,2.3380,1.1135,0.0,1.5,2.2,3.0,8.4
snow_depth_cm,8760.0,0.0796,0.4923,0.0,0.0,0.0,0.0,8.9
is_bad_weather,8760.0,0.1658,0.3719,0.0,0.0,0.0,0.0,1.0


In [29]:
# =========================
# 15-3. train / valid / test에 전일 차량운행 + 날씨 merge
# =========================


def add_prev_day_vehicle_and_weather_features(train_df, valid_df, test_df, prev_day_table, weather_table, weather_features):
    result_frames = []

    # train 기준 중앙값으로 결측 보완값 계산
    train_dates = train_df["접수일시"].dt.normalize()
    train_daily_values = train_dates.to_frame(name="기준일").merge(
        prev_day_table,
        on="기준일",
        how="left",
    )[PREV_DAY_VEHICLE_OPERATION_COL]
    prev_day_fallback = train_daily_values.median()

    train_weather_key = train_df[["접수일시"]].copy()
    train_weather_key["weather_date"] = train_weather_key["접수일시"].dt.date
    train_weather_key["weather_hour"] = train_weather_key["접수일시"].dt.hour
    train_weather_values = train_weather_key.merge(
        weather_table,
        on=["weather_date", "weather_hour"],
        how="left",
    )[weather_features]
    weather_fallback_values = train_weather_values.median(numeric_only=True)

    for name, frame in [
        ("train", train_df),
        ("valid", valid_df),
        ("test", test_df),
    ]:
        tmp = frame.copy()

        # 전일 차량운행 merge
        tmp["_merge_기준일"] = tmp["접수일시"].dt.normalize()
        tmp = tmp.merge(
            prev_day_table,
            left_on="_merge_기준일",
            right_on="기준일",
            how="left",
        )
        prev_day_missing_count = tmp[PREV_DAY_VEHICLE_OPERATION_COL].isna().sum()
        tmp[PREV_DAY_VEHICLE_OPERATION_COL] = tmp[PREV_DAY_VEHICLE_OPERATION_COL].fillna(prev_day_fallback)
        tmp = tmp.drop(columns=["_merge_기준일", "기준일"], errors="ignore")

        # 날씨 merge
        tmp["weather_date"] = tmp["접수일시"].dt.date
        tmp["weather_hour"] = tmp["접수일시"].dt.hour
        tmp = tmp.merge(
            weather_table,
            on=["weather_date", "weather_hour"],
            how="left",
        )
        weather_missing_counts = tmp[weather_features].isna().sum()
        tmp[weather_features] = tmp[weather_features].fillna(weather_fallback_values)

        print(f"{name}: shape={tmp.shape}")
        print(f"전일 차량운행 결측 보완 전: {prev_day_missing_count}")
        print("날씨 결측 보완 전:")
        display(weather_missing_counts.to_frame("missing_count"))

        result_frames.append(tmp)

    return result_frames


train_supply_weather, valid_supply_weather, test_supply_weather = add_prev_day_vehicle_and_weather_features(
    train,
    valid,
    test,
    vehicle_operation_prev_day_table,
    weather_merge_table,
    WEATHER_FEATURES_SIMPLE,
)

print("전일 차량운행 + 날씨 추가 후 shape:")
print("train_supply_weather:", train_supply_weather.shape)
print("valid_supply_weather:", valid_supply_weather.shape)
print("test_supply_weather:", test_supply_weather.shape)

display(
    train_supply_weather[
        ["접수일시", PREV_DAY_VEHICLE_OPERATION_COL] + WEATHER_FEATURES_SIMPLE + [TARGET_COL]
    ].head()
)

train: shape=(905434, 75)
전일 차량운행 결측 보완 전: 768
날씨 결측 보완 전:


,missing_count
temperature_c,0
precipitation_mm,0
wind_speed_ms,0
snow_depth_cm,0
is_bad_weather,0


valid: shape=(194022, 75)
전일 차량운행 결측 보완 전: 145
날씨 결측 보완 전:


,missing_count
temperature_c,0
precipitation_mm,0
wind_speed_ms,0
snow_depth_cm,0
is_bad_weather,0


test: shape=(194022, 75)
전일 차량운행 결측 보완 전: 147
날씨 결측 보완 전:


,missing_count
temperature_c,0
precipitation_mm,0
wind_speed_ms,0
snow_depth_cm,0
is_bad_weather,0


전일 차량운행 + 날씨 추가 후 shape:
train_supply_weather: (905434, 75)
valid_supply_weather: (194022, 75)
test_supply_weather: (194022, 75)


,접수일시,vehicle_operation_count_prev_day,temperature_c,precipitation_mm,wind_speed_ms,snow_depth_cm,is_bad_weather,target_min
0,2025-01-01 00:01:26.127,723.0,-1.2,0.0,0.7,0.0,0,48.447333
1,2025-01-01 00:07:46.960,723.0,-1.2,0.0,0.7,0.0,0,52.051550
2,2025-01-01 00:28:16.000,723.0,-1.2,0.0,0.7,0.0,0,38.206117
3,2025-01-01 00:28:20.117,723.0,-1.2,0.0,0.7,0.0,0,92.664717
4,2025-01-01 00:30:31.000,723.0,-1.2,0.0,0.7,0.0,0,82.073717


## 16. LGBM + 전일 차량운행 + 날씨 재학습

기존 Feature Set v2에 전일 차량운행과 날씨 피처를 추가해 LGBMRegressor를 다시 학습한다.

이 실험의 목적은 RF에서 효과가 있었던 공급·날씨 피처가 LGBM에서도 MAE 개선으로 이어지는지 확인하는 것이다.

In [30]:
# =========================
# 16-1. 전일 차량운행 + 날씨 Feature Set 정의
# =========================

FEATURES_V2_SUPPLY_WEATHER = FEATURES_V2 + [
    PREV_DAY_VEHICLE_OPERATION_COL,
] + WEATHER_FEATURES_SIMPLE

NUMERIC_COLS_V2_SUPPLY_WEATHER = NUMERIC_COLS_V2 + [
    PREV_DAY_VEHICLE_OPERATION_COL,
] + WEATHER_FEATURES_SIMPLE

missing_features = [col for col in FEATURES_V2_SUPPLY_WEATHER if col not in train_supply_weather.columns]
if missing_features:
    raise ValueError(f"train_supply_weather에 없는 피처가 있습니다: {missing_features}")

print("기존 피처 수:", len(FEATURES_V2))
print("전일 차량운행 + 날씨 추가 후 피처 수:", len(FEATURES_V2_SUPPLY_WEATHER))
print(FEATURES_V2_SUPPLY_WEATHER)

display(train_supply_weather[FEATURES_V2_SUPPLY_WEATHER + [TARGET_COL]].head())

기존 피처 수: 11
전일 차량운행 + 날씨 추가 후 피처 수: 17
['hour', '이용목적', '승차거리', '출발동', '목적동', '출발구', '목적구', 'month', '세부이동유형', 'dayofweek', 'model_group', 'vehicle_operation_count_prev_day', 'temperature_c', 'precipitation_mm', 'wind_speed_ms', 'snow_depth_cm', 'is_bad_weather']


,hour,이용목적,승차거리,출발동,목적동,출발구,목적구,month,세부이동유형,dayofweek,model_group,vehicle_operation_count_prev_day,temperature_c,precipitation_mm,wind_speed_ms,snow_depth_cm,is_bad_weather,target_min
0,0,기타,17667,남영동,수유제2동,용산구,강북구,1,구 간 이동,2,특장차_바로콜,723.0,-1.2,0.0,0.7,0.0,0,48.447333
1,0,기타,4858,상일동,성내제2동,강동구,강동구,1,구 내 이동,2,특장차_바로콜,723.0,-1.2,0.0,0.7,0.0,0,52.051550
2,0,귀가,5145,중계2.3동,상계1동,노원구,노원구,1,구 내 이동,2,특장차_바로콜,723.0,-1.2,0.0,0.7,0.0,0,38.206117
3,0,기타,9000,이화동,월계2동,종로구,노원구,1,구 간 이동,2,특장차_바로콜,723.0,-1.2,0.0,0.7,0.0,0,92.664717
4,0,귀가,9702,여의동,증산동,영등포구,은평구,1,구 간 이동,2,특장차_바로콜,723.0,-1.2,0.0,0.7,0.0,0,82.073717


In [31]:
# =========================
# 16-2. 전일 차량운행 + 날씨 LightGBM Pipeline 구성
# =========================


def build_lgbm_v2_supply_weather_pipeline(params=None):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                ),
                CATEGORICAL_COLS_V2,
            ),
            (
                "num",
                "passthrough",
                NUMERIC_COLS_V2_SUPPLY_WEATHER,
            ),
        ],
        verbose_feature_names_out=False,
        remainder="drop",
    )

    default_params = {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "objective": "regression",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    }

    if params:
        default_params.update(params)

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", LGBMRegressor(**default_params)),
    ])

In [32]:
# =========================
# 16-3. LGBMRegressor + 전일 차량운행 + 날씨 학습 및 평가
# =========================

lgbm_v2_supply_weather_model = build_lgbm_v2_supply_weather_pipeline()

lgbm_v2_supply_weather_model.fit(
    train_supply_weather[FEATURES_V2_SUPPLY_WEATHER],
    train_supply_weather[TARGET_COL],
)

valid_pred_lgbm_v2_supply_weather = lgbm_v2_supply_weather_model.predict(
    valid_supply_weather[FEATURES_V2_SUPPLY_WEATHER]
)
test_pred_lgbm_v2_supply_weather = lgbm_v2_supply_weather_model.predict(
    test_supply_weather[FEATURES_V2_SUPPLY_WEATHER]
)

valid_metrics_lgbm_v2_supply_weather = evaluate_predictions(
    valid_supply_weather,
    valid_pred_lgbm_v2_supply_weather,
    threshold=long_wait_threshold_v2,
)

test_metrics_lgbm_v2_supply_weather = evaluate_predictions(
    test_supply_weather,
    test_pred_lgbm_v2_supply_weather,
    threshold=long_wait_threshold_v2,
)

lgbm_v2_supply_weather_results = pd.DataFrame([
    {
        "model": "LightGBM Feature Set v2 + vehicle_operation_count_prev_day + weather",
        "split": "valid",
        **valid_metrics_lgbm_v2_supply_weather,
    },
    {
        "model": "LightGBM Feature Set v2 + vehicle_operation_count_prev_day + weather",
        "split": "test",
        **test_metrics_lgbm_v2_supply_weather,
    },
])

display(
    lgbm_v2_supply_weather_results
    .drop(columns=["risk_Confusion_Matrix"], errors="ignore")
    .round(4)
)

print("test confusion matrix:")
test_metrics_lgbm_v2_supply_weather["risk_Confusion_Matrix"]

,model,split,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,LightGBM Feature Set v2 + vehicle_operation_co...,valid,11.4268,8.4439,15.5988,0.5891,75.6889,0.9226,0.7503,0.3377,0.4658
1,LightGBM Feature Set v2 + vehicle_operation_co...,test,11.4456,8.4885,15.6172,0.5861,75.6889,0.9230,0.7488,0.3353,0.4632


test confusion matrix:


[[172628, 2163], [12782, 6449]]

## 17. LGBM 전체 실험 결과 비교

기존 LGBM, p50 proxy 추가 LGBM, 전일 차량운행+날씨 추가 LGBM을 비교한다.

해석 기준은 MAE를 최우선으로 보고, Median_AE, RMSE, R2, risk_F1을 보조적으로 확인한다.

In [33]:
# =========================
# 17-1. LGBM baseline / p50 / 전일차량운행+날씨 비교
# =========================

lgbm_compare_frames = [
    lgbm_v2_results.assign(feature_set="Feature Set v2"),
]

if "lgbm_v2_p50_results" in globals():
    lgbm_compare_frames.append(
        lgbm_v2_p50_results.assign(feature_set="Feature Set v2 + origin_gu_weekday_hour_wait_p50")
    )

lgbm_compare_frames.append(
    lgbm_v2_supply_weather_results.assign(feature_set="Feature Set v2 + vehicle_operation_count_prev_day + weather")
)

lgbm_feature_compare = pd.concat(lgbm_compare_frames, ignore_index=True)

compare_cols = [
    "feature_set",
    "model",
    "split",
    "MAE",
    "Median_AE",
    "RMSE",
    "R2",
    "risk_threshold",
    "risk_Accuracy",
    "risk_Precision",
    "risk_Recall",
    "risk_F1",
]

display(
    lgbm_feature_compare[compare_cols]
    .sort_values(["split", "MAE"])
    .round(4)
)

baseline_test = lgbm_v2_results.query("split == 'test'").iloc[0]
supply_weather_test = lgbm_v2_supply_weather_results.query("split == 'test'").iloc[0]

summary_rows = []
for metric in ["MAE", "Median_AE", "RMSE", "R2", "risk_F1"]:
    summary_rows.append({
        "metric": metric,
        "baseline": baseline_test[metric],
        "supply_weather": supply_weather_test[metric],
        "diff_supply_weather_minus_baseline": supply_weather_test[metric] - baseline_test[metric],
    })

if "lgbm_v2_p50_results" in globals():
    p50_test = lgbm_v2_p50_results.query("split == 'test'").iloc[0]
    for metric in ["MAE", "Median_AE", "RMSE", "R2", "risk_F1"]:
        summary_rows.append({
            "metric": metric,
            "baseline": p50_test[metric],
            "supply_weather": supply_weather_test[metric],
            "diff_supply_weather_minus_baseline": supply_weather_test[metric] - p50_test[metric],
            "comparison_base": "p50_proxy",
        })

lgbm_supply_weather_improvement_summary = pd.DataFrame(summary_rows)

display(lgbm_supply_weather_improvement_summary.round(4))

,feature_set,model,split,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
5,Feature Set v2 + vehicle_operation_count_prev_...,LightGBM Feature Set v2 + vehicle_operation_co...,test,11.4456,8.4885,15.6172,0.5861,75.6889,0.9230,0.7488,0.3353,0.4632
3,Feature Set v2 + origin_gu_weekday_hour_wait_p50,LightGBM Feature Set v2 + origin_gu_weekday_ho...,test,11.8465,8.6042,16.3239,0.5478,75.6889,0.9198,0.6992,0.3353,0.4533
1,Feature Set v2,LightGBM Feature Set v2,test,11.9504,8.7772,16.3911,0.5441,75.6889,0.9190,0.7155,0.3032,0.4259
4,Feature Set v2 + vehicle_operation_count_prev_...,LightGBM Feature Set v2 + vehicle_operation_co...,valid,11.4268,8.4439,15.5988,0.5891,75.6889,0.9226,0.7503,0.3377,0.4658
2,Feature Set v2 + origin_gu_weekday_hour_wait_p50,LightGBM Feature Set v2 + origin_gu_weekday_ho...,valid,11.8254,8.6074,16.2798,0.5525,75.6889,0.9199,0.7087,0.3368,0.4566
0,Feature Set v2,LightGBM Feature Set v2,valid,11.9254,8.7532,16.3528,0.5485,75.6889,0.9190,0.7215,0.3079,0.4316


,metric,baseline,supply_weather,diff_supply_weather_minus_baseline,comparison_base
0,MAE,11.9504,11.4456,-0.5048,NaN
1,Median_AE,8.7772,8.4885,-0.2887,NaN
2,RMSE,16.3911,15.6172,-0.7739,NaN
3,R2,0.5441,0.5861,0.0420,NaN
4,risk_F1,0.4259,0.4632,0.0373,NaN
5,MAE,11.8465,11.4456,-0.4009,p50_proxy
6,Median_AE,8.6042,8.4885,-0.1156,p50_proxy
7,RMSE,16.3239,15.6172,-0.7067,p50_proxy
8,R2,0.5478,0.5861,0.0383,p50_proxy
9,risk_F1,0.4533,0.4632,0.0099,p50_proxy


In [34]:
# =========================
# 17-2. 전일 차량운행 + 날씨 모델 차량유형별 성능 평가
# =========================

group_metrics_lgbm_v2_supply_weather = evaluate_by_group(
    test_supply_weather,
    test_pred_lgbm_v2_supply_weather,
    threshold_by_group_v2,
)

display(group_metrics_lgbm_v2_supply_weather.round(4))

,model_group,rows,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,임차택시_바로콜,41738,10.2699,7.6291,14.0191,0.5741,66.0455,0.9228,0.6973,0.3585,0.4735
1,특장차_바로콜,152284,11.7678,8.7561,16.0274,0.5825,78.4430,0.9215,0.7529,0.3067,0.4358


## 18. 해석 메모

이 실험은 RandomForest에서 성능 개선이 확인된 `전일 차량운행 + 날씨` 피처가 LGBM에서도 효과가 있는지 확인하기 위한 것이다.

발표에서는 다음 기준으로 해석한다.

- 기존 LGBM 대비 MAE가 감소하면 공급·날씨 피처가 LGBM에서도 평균 대기시간 예측에 도움이 된 것이다.
- p50 proxy보다 MAE가 낮으면 target 기반 proxy 없이도 예측 시점에 사용 가능한 외부 피처만으로 더 좋은 성능을 낸 것으로 해석할 수 있다.
- MAE 개선이 작더라도 risk_F1이 증가하면 장시간 대기 탐지 관점에서 보조 효과가 있다고 볼 수 있다.